Importing Dependencies

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import shutil

In [3]:
!kaggle

usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,config,auth}
              ...
kaggle: error: the following arguments are required: command


In [4]:
# configuring the path of Kaggle.json file
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [5]:
!kaggle datasets download puneet6060/intel-image-classification

Dataset URL: https://www.kaggle.com/datasets/puneet6060/intel-image-classification
License(s): copyright-authors
intel-image-classification.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
!ls

intel-image-classification.zip	sample_data  seg_test	train_images
kaggle.json			seg_pred     seg_train


In [7]:
from zipfile import ZipFile
dataset = '/content/intel-image-classification.zip'

with ZipFile(dataset, 'r') as zip:
  zip.extractall()
  print('The dataset is extracted')

The dataset is extracted


In [8]:
!ls

intel-image-classification.zip	sample_data  seg_test	train_images
kaggle.json			seg_pred     seg_train


In [9]:
train = '/content/seg_train/seg_train'
test = '/content/seg_test/seg_test'

In [10]:
train_n = os.listdir(train)
test_n = os.listdir(test)

In [11]:
print(train_n)
print(test_n)

['mountain', 'street', 'glacier', 'forest', 'sea', 'buildings']
['mountain', 'street', 'glacier', 'forest', 'sea', 'buildings']


In [12]:
# Iterate through each category in the test dataset
for category_name in train_n:
    category_path = os.path.join(train, category_name)
    files_in_category = os.listdir(category_path)
    print(f"Category: {category_name}, Number of files: {len(files_in_category)}")
    # Optionally, print the first few files in each category
    # print(f"  First 5 files: {files_in_category[:5]}")

Category: mountain, Number of files: 2512
Category: street, Number of files: 2382
Category: glacier, Number of files: 2404
Category: forest, Number of files: 2271
Category: sea, Number of files: 2274
Category: buildings, Number of files: 2191


In [13]:
import tensorflow as tf

testset = tf.keras.utils.image_dataset_from_directory(
    "/content/seg_test/seg_test",
    image_size=(150,150),
    batch_size=32,
    label_mode='categorical'
)

Found 3000 files belonging to 6 classes.


In [14]:
test_images_list = []
test_labels_list = []

for images, labels in testset:
    test_images_list.append(images)
    test_labels_list.append(labels)

In [15]:
X_test = np.concatenate(test_images_list)
Y_test = np.concatenate(test_labels_list)

print(X_test.shape)   # (14034, 150, 150, 3)
print(Y_test.shape)   # (14034, 6)

(3000, 150, 150, 3)
(3000, 6)


In [16]:
import tensorflow as tf

dataset = tf.keras.utils.image_dataset_from_directory(
    "/content/seg_train/seg_train",
    image_size=(150,150),
    batch_size=32,
    label_mode='categorical'
)

Found 14034 files belonging to 6 classes.


In [17]:
images_list = []
labels_list = []

for images, labels in dataset:
    images_list.append(images)
    labels_list.append(labels)

In [18]:
X_train = np.concatenate(images_list)
Y_train = np.concatenate(labels_list)

print(X_train.shape)   # (14034, 150, 150, 3)
print(Y_train.shape)   # (14034, 6)

(14034, 150, 150, 3)
(14034, 6)


In [19]:
print(X_train.shape)
print(Y_train.shape)

(14034, 150, 150, 3)
(14034, 6)


In [20]:
X_train[0]

array([[[ 25.,  54.,   0.],
        [ 63.,  94.,  35.],
        [ 48.,  83.,  19.],
        ...,
        [148., 180.,  97.],
        [115., 141.,  80.],
        [ 99., 123.,  71.]],

       [[ 14.,  43.,   0.],
        [119., 150.,  93.],
        [ 20.,  56.,   0.],
        ...,
        [125., 154.,  74.],
        [120., 146.,  83.],
        [ 87., 112.,  57.]],

       [[ 85., 115.,  61.],
        [ 38.,  73.,  19.],
        [ 19.,  58.,   5.],
        ...,
        [129., 157.,  80.],
        [130., 157.,  90.],
        [ 85., 111.,  50.]],

       ...,

       [[  5.,  22.,   0.],
        [ 37.,  54.,  22.],
        [ 46.,  62.,  35.],
        ...,
        [ 38.,  40.,  29.],
        [ 18.,  18.,  10.],
        [  1.,   1.,   0.]],

       [[  0.,  15.,   0.],
        [ 45.,  61.,  34.],
        [ 41.,  57.,  31.],
        ...,
        [ 14.,  13.,   8.],
        [ 18.,  14.,  13.],
        [ 15.,  11.,  10.]],

       [[ 63.,  82.,  54.],
        [ 30.,  48.,  22.],
        [ 21.,  

Building the Neural Network


In [27]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow import keras


In [28]:
model = Sequential([
    layers.Rescaling(1./255, input_shape=(150,150,3)),
    layers.Conv2D(32, (3,3) ,activation = 'relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3) ,activation = 'relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(6, activation='softmax')
])

In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36992)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,735,104 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,829,126 (18.42 MB)

 Trainable params: 4,829,126 (18.42 MB)

 Non-trainable params: 0 (0.00 B)

In [30]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    X_train, Y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=32
)

In [ ]:
model.evaluate(X_test,Y_test)

In [ ]:
h = history

#plot the loss value
plt.plot(h.history['loss'], label='train loss')
plt.plot(h.history['val_loss'], label = 'validation loss')
plt.legend()
plt.show()
#plot the accuracy value
plt.plot(h.history['accuracy'], label='train accuracy')
plt.plot(h.history['val_accuracy'], label = 'validation accuracy')
plt.legend()
plt.show()

In [ ]:
pred = model.predict(X_test)

In [ ]:
y_pred = np.argmax(pred, axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(Y_test, y_pred)
cr = classification_report(Y_test, y_pred)
print(cm)
print(cr)